### CEO Readiness Check

Final gate in the CEO initializer job. Polls until:
- All 5 Knowledge Assistants (legal, regulatory, audits, consultancy, inspection) are ONLINE and have finished syncing their documents
- The CEO Supervisor (MAS) serving endpoint is READY
- All 5 UC volumes contain at least one PDF

Fails the job if any resource is not ready within the timeout.

In [ ]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

In [ ]:
import time, json as _json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
KA_API  = "/api/2.0/knowledge-assistants"
MAS_API = "/api/2.0/multi-agent-supervisors"
EP_API  = "/api/2.0/serving-endpoints"

POLL_INTERVAL_S = 20
MAX_POLLS       = 60

EXPECTED_KA_NAMES = [
    f"{CATALOG}-ceo-legal",
    f"{CATALOG}-ceo-regulatory",
    f"{CATALOG}-ceo-audits",
    f"{CATALOG}-ceo-consultancy",
    f"{CATALOG}-inspection-knowledge",
]
MAS_NAME = f"{CATALOG}-ceo-supervisor"


def _paginate(api_path, list_key):
    items, params = [], {}
    try:
        while True:
            resp = w.api_client.do("GET", api_path, query=params)
            found = resp.get(list_key)
            if found is None:
                found = next((v for v in resp.values() if isinstance(v, list)), [])
            items.extend(found or [])
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception as e:
        print(f"  \u26a0\ufe0f  {api_path} error: {e}")
        if items:
            print(f"       Partial results only \u2014 {len(items)} item(s) retrieved before failure; IDs will be re-validated")
    return items

def _list_kas():    return _paginate(KA_API, "knowledge_assistants")
def _list_mas():    return _paginate(MAS_API, "multi_agent_supervisors")
def _list_tiles():  return _paginate("/api/2.0/tiles", "tiles")

def _ka_id_from_item(item):
    ka = item.get("knowledge_assistant", item)
    return item.get("tile_id") or ka.get("tile_id") or ka.get("id")


# ── Source 1: uc_state (KA notebook writes correct IDs here) ──────────────────
ka_by_name = {}
mas_endpoint = ""
try:
    rows = spark.sql(f"""
        SELECT resource_type, resource_data FROM {CATALOG}._internal_state.resources
        WHERE resource_type IN ('knowledge_assistants', 'multi_agent_supervisors')
        ORDER BY created_at DESC
    """).collect()
    seen = set()
    for row in rows:
        info = _json.loads(row.resource_data)
        name = info.get("name", "")
        tid  = info.get("tile_id", "")
        if row.resource_type == "knowledge_assistants" and name in EXPECTED_KA_NAMES and name not in seen and tid:
            ka_by_name[name] = tid
            seen.add(name)
        elif row.resource_type == "multi_agent_supervisors" and not mas_endpoint:
            mas_endpoint = info.get("endpoint_name", "")
    print(f"uc_state: {len(ka_by_name)} KA(s), MAS endpoint={mas_endpoint or '(none)'}")
except Exception as e:
    print(f"uc_state unavailable: {e}")

# ── Source 2: KA API (prints full names so we can see what's there) ────────────
all_kas = _list_kas()
print(f"\nKA API returned {len(all_kas)} item(s)")
for item in all_kas:
    ka = item.get("knowledge_assistant", item)
    print(f"  name={ka.get('name')!r}  tile_id={(_ka_id_from_item(item))!r}  ep={ka.get('endpoint_name')!r}")
    if ka.get("name") in EXPECTED_KA_NAMES and ka.get("name") not in ka_by_name:
        kid = _ka_id_from_item(item)
        if kid:
            ka_by_name[ka.get("name")] = kid

# ── Source 3: tiles API ────────────────────────────────────────────────────────
all_tiles = _list_tiles()
print(f"\nTiles API returned {len(all_tiles)} item(s)")
for tile in all_tiles:
    n = tile.get("name", "")
    if n in EXPECTED_KA_NAMES or n == MAS_NAME:
        tid = tile.get("tile_id") or tile.get("id")
        ep  = tile.get("serving_endpoint_name") or tile.get("endpoint_name", "")
        print(f"  name={n!r}  tile_id={tid!r}  ep={ep!r}")
        if n in EXPECTED_KA_NAMES and n not in ka_by_name and tid:
            ka_by_name[n] = tid
        if n == MAS_NAME and not mas_endpoint and ep:
            mas_endpoint = ep

# ── MAS: also try MAS API ──────────────────────────────────────────────────────
if not mas_endpoint:
    for item in _list_mas():
        mas = item.get("multi_agent_supervisor", item)
        if mas.get("name") == MAS_NAME:
            mas_endpoint = mas.get("endpoint_name", "")
            break

# ── Validate collected KA IDs — UC state can be stale after KA recreation ─────
# Call GET /{id} for each resolved ID; any that 404 are stale and need to be
# re-resolved from the tiles API (which always reflects current state).
_stale_kas = []
for _name, _tid in list(ka_by_name.items()):
    try:
        w.api_client.do("GET", f"{KA_API}/{_tid}")
    except Exception:
        print(f"  ⚠️  {_name}: tile_id {_tid!r} is stale — will re-resolve via Tiles API")
        _stale_kas.append(_name)

if _stale_kas:
    for _tile in _list_tiles():
        _n = _tile.get("name", "")
        if _n in _stale_kas:
            _new_tid = _tile.get("tile_id") or _tile.get("id")
            if _new_tid:
                ka_by_name[_n] = _new_tid
                _stale_kas.remove(_n)
                print(f"  ✅ Re-resolved {_n} → {_new_tid}")
    if _stale_kas:
        raise RuntimeError(
            f"Could not re-resolve stale KA IDs via Tiles API: {_stale_kas}\n"
            "Re-create the affected KAs via the stage notebooks and re-run."
        )

print(f"\nKAs resolved ({len(ka_by_name)}/{len(EXPECTED_KA_NAMES)}):")
for name, tid in ka_by_name.items():
    print(f"  {name} → {tid}")
missing = [n for n in EXPECTED_KA_NAMES if n not in ka_by_name]
if missing:
    print(f"  ⚠️  Not found: {missing}")
print(f"MAS endpoint: {mas_endpoint or '(not found)'}")


In [ ]:
# ── Volume file checks ────────────────────────────────────────────────────────
VOLUMES = [
    ("Legal Complaints",    f"/Volumes/{CATALOG}/legal_complaints/documents"),
    ("Regulatory Docs",     f"/Volumes/{CATALOG}/regulatory/documents"),
    ("Audit Reports",       f"/Volumes/{CATALOG}/audits/reports"),
    ("Consultancy Reports", f"/Volumes/{CATALOG}/consultancy/reports"),
    ("Inspection Reports",  f"/Volumes/{CATALOG}/food_safety/reports"),
]

volume_errors = []
for label, path in VOLUMES:
    try:
        files = [f for f in dbutils.fs.ls(path) if f.name.endswith(".pdf")]
        print(f"✅ {label}: {len(files)} PDFs at {path}")
    except Exception as e:
        msg = f"{label}: {path} — {e}"
        print(f"❌ {msg}")
        volume_errors.append(msg)

if volume_errors:
    raise RuntimeError(f"Volume check failed:\n" + "\n".join(volume_errors))

print("\n✅ All volumes have PDFs")

In [ ]:
# ── Parallel readiness poll (all KAs + MAS simultaneously) ─────────────────
import concurrent.futures

POLL_INTERVAL_S = 20
MAX_POLLS       = 30   # 10 min max — enough for any cold-start scenario

if not ka_by_name:
    raise RuntimeError(
        f"No KAs found. Expected: {EXPECTED_KA_NAMES}\nCheck the diagnostic output above."
    )
if not mas_endpoint:
    raise RuntimeError("MAS endpoint not found — ensure CEO_Supervisor ran successfully")

_poll_deadline = time.time() + POLL_INTERVAL_S * MAX_POLLS
_results: dict = {}  # resource name → True/False


def _poll_ka(name: str, tile_id: str):
    for attempt in range(1, MAX_POLLS + 1):
        if time.time() > _poll_deadline:
            print(f"  \u23f1  {name} — shared deadline reached")
            return name, False
        try:
            resp = w.api_client.do("GET", f"{KA_API}/{tile_id}")
            ka_status   = resp.get("knowledge_assistant", {}).get("status", {})
            ep_status   = str(ka_status.get("endpoint_status", "")).upper()
            sync_status = str(ka_status.get("sync_status",
                               ka_status.get("indexing_status", ""))).upper()
            ep_ready   = ep_status in ("ONLINE", "ACTIVE", "READY")
            sync_ready = (not sync_status) or sync_status in (
                "SYNCED", "READY", "COMPLETE", "SUCCESS", "DONE", ""
            )
            if ep_ready and sync_ready:
                print(f"  \u2705 {name} READY (endpoint={ep_status}, sync={sync_status or 'n/a'})")
                return name, True
            print(f"  \u23f3 [{attempt}/{MAX_POLLS}] {name} endpoint={ep_status}, sync={sync_status or 'n/a'}")
        except Exception as e:
            print(f"  \u26a0\ufe0f  {name} poll error: {e} — re-resolving")
            _re_resolved = False
            for item in _list_kas():
                ka = item.get("knowledge_assistant", item)
                if ka.get("name") == name:
                    new_id = _ka_id_from_item(item)
                    if new_id and new_id != tile_id:
                        print(f"    Re-resolved via KA API: {new_id}")
                        tile_id = new_id
                    _re_resolved = True
                    break
            if not _re_resolved:
                for _tile in _list_tiles():
                    if _tile.get("name") == name:
                        new_id = _tile.get("tile_id") or _tile.get("id")
                        if new_id and new_id != tile_id:
                            print(f"    Re-resolved via Tiles API: {new_id}")
                            tile_id = new_id
                        break
        remaining = _poll_deadline - time.time()
        if remaining <= 0:
            return name, False
        time.sleep(min(POLL_INTERVAL_S, remaining))
    return name, False


def _poll_mas(endpoint: str):
    print(f"  Polling MAS endpoint: {endpoint}")
    for attempt in range(1, MAX_POLLS + 1):
        if time.time() > _poll_deadline:
            print(f"  \u23f1  MAS — shared deadline reached")
            return "MAS", False
        try:
            resp = w.api_client.do("GET", f"{EP_API}/{endpoint}")
            state  = resp.get("state", {})
            ready  = str(state.get("ready", "")).upper()
            config_update = str(state.get("config_update", "")).upper()
            if ready == "READY":
                print(f"  \u2705 MAS READY (config_update={config_update or 'n/a'})")
                return "MAS", True
            print(f"  \u23f3 [{attempt}/{MAX_POLLS}] MAS ready={ready}, config_update={config_update}")
        except Exception as e:
            print(f"  \u26a0\ufe0f  MAS poll error: {e}")
        remaining = _poll_deadline - time.time()
        if remaining <= 0:
            return "MAS", False
        time.sleep(min(POLL_INTERVAL_S, remaining))
    return "MAS", False


print(f"Polling {len(ka_by_name)} KAs + MAS in parallel (max {MAX_POLLS * POLL_INTERVAL_S // 60} min)…\n")

with concurrent.futures.ThreadPoolExecutor(max_workers=len(ka_by_name) + 1) as _pool:
    _futures = {_pool.submit(_poll_ka, name, tid): name for name, tid in ka_by_name.items()}
    _futures[_pool.submit(_poll_mas, mas_endpoint)] = "MAS"
    for _fut in concurrent.futures.as_completed(_futures):
        _resource, _ok = _fut.result()
        _results[_resource] = _ok


In [ ]:
# ── Check all resources passed ───────────────────────────────────────────────
failures = [r for r, ok in _results.items() if not ok]
if failures:
    raise RuntimeError(
        f"Resources not ready after {MAX_POLLS * POLL_INTERVAL_S // 60} min: {failures}\n"
        "Check the logs above for per-resource status."
    )


In [ ]:
print("=" * 55)
print("✅  CEO READINESS CHECK PASSED")
print("=" * 55)
print(f"  KAs ready:       {len(ka_by_name)}")
print(f"  MAS endpoint:    {mas_endpoint}")
print(f"  Volumes checked: {len(VOLUMES)}")
print("\nThe CEO Dashboard is ready for use.")